In [1]:
import sys
original_sys_path = sys.path.copy()
sys.path.append('../')
from utils_models import *

dq.set_device('cpu')
import warnings
warnings.filterwarnings("ignore")

In [2]:
solver = dq.solver.Dopri8(
                    rtol= 1e-06,
                    atol= 1e-06,
                    safety_factor= 0.6,
                    min_factor= 0.1,
                    max_factor = 4.0,
                    max_steps = int(1e4*1000),
                )

n_lvls_fluxonium = 20
n_lvls_transmon = 4


Ej_f = 2.7
Ec_f = 0.6
El_f = 0.13
qsf = qs.Fluxonium.create(
    n_lvls_fluxonium,
    {"Ej": Ej_f, "Ec": Ec_f, "El": El_f, "phi_ext": 0.0},
    N_pre_diag=100,
    use_linear = False
    )
g_tf = 0.2
Ec_t = 0.2


def truncate(data: jnp.array):
    return data[:,:]

tot_dim = n_lvls_fluxonium * n_lvls_transmon


In [10]:

def objective(params):    
    sigma = params[0]
    amp_with_2pi = params[1]
    Ej_t = params[2]
    w_d = params[3]
    beta = params[4]
    
    qst = MyTransmon.create(
        N = n_lvls_transmon,
        params = {"Ej": Ej_t, "Ec": Ec_t,"ng":0.0},
        N_max_charge=10
        )
    

    devices = [qsf, qst]
    f_indx = 0
    t_indx = 1
    Ns = [device.N for device in devices]
    fn = qs.promote(qsf.ops["n"], f_indx, Ns)
    tn = qs.promote(qst.ops['n'], t_indx, Ns)

    system = qs.System.create(devices, couplings=[
        g_tf *  fn @ tn
        ])
    system.params["g_tf"] = g_tf
    system_evals_sorted, system_evecs_sorted, product_indices_sorted_by_eval = calculate_eig(Ns, system.get_H())
    driven_op = transform_op_into_dressed_basis_jax(tn, system_evecs_sorted.T)
    # system_evals_in_product_indices, system_evecs_in_product_indices = system.calculate_eig_linear()
    # w_d = system_evals_in_product_indices[0,1] - system_evals_in_product_indices[0,0]


    pulse_length = 6*sigma
    pulse_shape_args={
        'w_d': w_d,
        'amp': amp_with_2pi/(2*jnp.pi),
        'duration': pulse_length,
        'sigma': sigma,
        'beta':beta
    }      

    tlist = jnp.linspace(0,pulse_length,int(pulse_length))

    def _H(t):
        _H =  2 * jnp.pi *truncate(jnp.diag(system_evals_sorted))
        _H += truncate(driven_op) * modified_drag_pulse(t, pulse_shape_args)
        return _H 
    H =  dq.timecallable(_H)
    
    psi0_list = [truncate(dq.basis(tot_dim,find_closest_dressed_index(l*qst.N, product_indices_sorted_by_eval)))
                        for l in [0,1,2]] #00,10,20
    result = dq.sesolve(
                H = H,
                psi0 = psi0_list,
                tsave = tlist,
                solver = solver
                )
        

    rho0 = dq.todm(result.states[0][-1]) / dq.trace(dq.todm(result.states[0][-1])).real
    rho1 = dq.todm(result.states[1][-1]) / dq.trace(dq.todm(result.states[1][-1])).real
    rho2 = dq.todm(result.states[2][-1]) / dq.trace(dq.todm(result.states[2][-1])).real

    f0_e = dq.expect(dq.basis_dm(tot_dim, find_closest_dressed_index(0*qst.N+1, product_indices_sorted_by_eval)),
                    rho0).real
    
    f1_e = dq.expect(dq.basis_dm(tot_dim, find_closest_dressed_index(1*qst.N+1, product_indices_sorted_by_eval)),
                    rho1).real
    
    f2_e = dq.expect(dq.basis_dm(tot_dim, find_closest_dressed_index(2*qst.N+1, product_indices_sorted_by_eval)),
                    rho2).real
    
    return 1 - f0_e + f1_e + f2_e




In [11]:

func = jax.value_and_grad(objective)
params = jnp.array([ 9.70272812e+01,  9.15152087e-03,  3.40891152e+01,  7.17587778e+00,-5.06040151e-04])
optimizer = optax.nadam(learning_rate=jnp.array([1.0,
                                                 1e-3,
                                                 1e-3,
                                                 1e-4,
                                                 1e-3])) 
opt_state = optimizer.init( params )

num_steps = 1000
for step in range(num_steps):
    print(f"iter: {step}, params: {params}")
    val, grads = func(params)
    print(f"\t val={val} grads: {grads}")
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)

print(f'Optimized params: {params}')

iter: 0, params: [ 9.70272812e+01  9.15152087e-03  3.40891152e+01  7.17587778e+00
 -5.06040151e-04]
	 val=0.0035151611622711893 grads: [-1.55887598e-03 -1.68157724e+01 -3.53878530e+00  3.82219152e+01
  2.61468938e-06]
iter: 1, params: [ 9.85009560e+01  1.06252051e-02  3.40905889e+01  7.17573041e+00
 -1.97410966e-03]
	 val=0.0701636501690361 grads: [ 8.16244812e-03  7.18515134e+01  1.70610692e+01 -1.84506944e+02
 -1.73427820e-04]
iter: 2, params: [ 9.73880747e+01  9.53972418e-03  3.40894863e+01  7.17584069e+00
 -7.67031928e-04]
	 val=0.0005859635477679552 grads: [ 6.49875262e-04  6.65586809e+00  1.59578789e+00 -1.72287359e+01
  6.31325356e-07]
iter: 3, params: [ 9.69720702e+01  9.12929400e-03  3.40890631e+01  7.17588300e+00
 -3.68675586e-04]
	 val=0.004416100644635824 grads: [-1.68721924e-03 -1.83002728e+01 -4.24407481e+00  4.57921025e+01
  3.76434120e-06]
iter: 4, params: [ 9.68614145e+01  9.07504014e-03  3.40889944e+01  7.17588994e+00
 -4.11475847e-05]
	 val=0.006380385759532959 grads